# Local run of the WOFOST 8.0 API app
***
**Author**: Mattia C. Mancini (m.c.mancini@exeter.ac.uk)  
**Affiliation**: LEEP Institute, University of Exeter  
**Date**: November 18th, 2025  
***  
This notebook illustrates how to run a local instance of the UK implementation of the WOFOST 8.0 crop yield model API app for a specified parcel, crop, and with default or user-modified input parameters.  
The notebook will go through the following steps:  
1. [Start the app](#1-start-a-local-instance-of-the-app)
1. [Specify a location for which the model needs to be run.]()
2. [Generate an instance of the WOFOST simulator for that location.]()
3. [Specify a crop and its management]()
4. [Pass the crop and management to the simulator to return yields.]()

## 1. Start a local instance of the app

### What is the WOFOST API?

The WOFOST API is a **web service** that allows you to run crop yield simulations programmatically. Instead of calling Python functions directly, you send requests to the API over HTTP, and it returns the simulation results. This approach offers several advantages:

- **Separation of concerns**: The simulation logic runs independently from your analysis code
- **Remote execution**: You can run simulations on a more powerful server while working locally
- **Multiple clients**: Different applications (R, Python, web browsers) can all use the same API
- **Scalability**: The API can handle multiple simultaneous requests

### How does it work?

The API is built using **FastAPI**, a modern Python web framework, and runs using **Uvicorn**, a lightning-fast web server. When active, the API:

1. Listens for incoming HTTP POST requests on specific **endpoints** (URLs)
2. Receives simulation parameters (parcel ID, crop type, year, etc.) as JSON data
3. Runs the WOFOST model with those parameters
4. Returns the results as JSON

There are currently two available endpoints:
- `/run_crop` - Run a single crop simulation. This is primarily for testing purposes and will run WOFOST on a specified crop for a specified year, with default parameters and agromanagement.
- `/run_bulk` - This is the main endpoint to use, allowing to run WOFOST for crops with custom parameters and managment, as well as parallel execution of multiple WOFOST runs.  


### Starting the API

To activate the API on your local machine, you need to open a **separate terminal** window and run:

```bash
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```
This keeps the API running in the background while you work through the notebook.
If the API starts successfully, your terminal will show something like:
```bash
INFO:     Will watch for changes in these directories: ['D:\\Documents\\GitHub\\UkWofost']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [15236] using WatchFiles
```
This means the API is active and listening for requests; next, we need to build POST requests that are then formatted and passed to the required endpoint.  The procedures to build POST requests follow the steps outlined at the start of this document 

## 2. Build the payload to pass to the API endpoint.
We will focus here on the `run_bulk` endpoint as it will be the primary endpoint to use.  
The expected payload is a JSON formatted structure that looks like the following:  
```
{
    "runs":
    [
        {
            "crop": "spring_barley",
            "variety": "Spring_barley_301",
            "year": 2018,
            "start_crop_calendar": "17/09/2017",
            "crop_start_date": "06/04/2018",
            "parcel_id": 5646522,
            "WAV": 34.4563,
            "SMLIM": 0.324,
            "CRAIRC": 0.0664,
            "WILTING_POTENTIAL": 3,
            "FIELD_CAPACITY": -0.2776,
            "SOPE": 1.559,
            "KSUB": 4.7259,
            "RDMSOL": 86.4714,
            "NAVAILI": 72.3212,
            "PAVAILI": 24.746,
            "KAVAILI": 39.1097,
            "N_1": 143.2409,
            "N_2": 138.3827,
            "N_3": 22.8097,
            "P_1": 42.2561,
            "P_2": 40.8229,
            "P_3": 6.7289,
            "K_1": 35.3805,
            "K_2": 34.1805,
            "K_3": 5.634,
            "NPK_T1": "06/04/2018",
            "NPK_T2": "26/06/2018",
            "NPK_T3": "21/07/2018"
        }
    ]
}
```
where:
- `runs` is a list containing one or multiple individual WOFOST runs with user defined paramenter. Each item in the list is a Python dictionary with key-value pairs representing input parameters. The minimum required parameters for a run are the following:  
    1.  `crop`: the crop of interest
    2. `variety`: the crop variety of interest
    3.  `year`: the calendar year of interest. For spring crops the calendar year is the same as the year in which the crop is panted and harvested; for winter crops, the calendar year is the year in which harvest happens (so year of planting + 1)
    4. `start_crop_calendar`: the date at which the crop calendar starts. This is either the same as the planting date, or happens before. If it is declared before the planting date, WOFOST will start tracking state variables from the date in which the crop calendar begins assuming that the soil is bare, up until the date in which the crop is planted.
    5. `crop_start_date`: the date in which the crop is planted
    6. `parcel_id`: the unique integer identifier of a parcel of interest; this comes from the vector land parcels in the [CEH Vector Land Cover Map](https://www.ceh.ac.uk/data/ukceh-land-cover-maps).



In [ ]:
# Update sys path so notebook can access ukwofost package
import sys
sys.path.append('../')

In [ ]:
# Build a payload to sent to the local API endpoint
payload = {
    "runs":
    [
        {
            "crop": "spring_barley",
            "variety": "Spring_barley_301",
            "year": 2018,
            "start_crop_calendar": "17/09/2017",
            "crop_start_date": "06/04/2018",
            "parcel_id": 5646522,
        }
    ]
}

In addition to specifying the payload, a summary flag needs to be declared. This tells the API what kind of output needs to be returned after a call. There are three options:  
1. `summary`: this returns the yield of the crop in kg at standard moisture at harvest for each run
2. `full`: this returns the time series of all state variables of WOFOST for each run from the starting date of the crop calendar to the date in which the crop is harvested
3. `harvest`: this returns all state variables of WOFOST at the day of harvest for each crop  

In [ ]:
SUMMARY_FLAG = 'harvest' 

Once the payload and the summary flasgs have been declared, we can then construct the URL of the endpoint. This is done as follows:

In [ ]:
ENDPOINT = "run_bulk"
URL = f"http://localhost:8000/{ENDPOINT}?summary={SUMMARY_FLAG}"

We can then send a post request to the API passing the URL and the payload using the `requests` package and wrap the response into a pandas dataframe:

In [ ]:
import pandas as pd
import requests
response = requests.post(URL, json=payload)
df = pd.DataFrame(response.json()["result"])

This run for spring barley in 2018 results in a yield of 317kg DM harvested on the 26th of July, 2018:

In [ ]:
print(df.day)
print(df.TWSO)

Any crop, soil or management parameter can be defined and passed to the API as a JSON payload.  
Below we are altering the previous run for spring barley adding three fertilisation events, which are defined by their timings and fertiliser quantities in kg:

In [ ]:
payload["runs"][0]["N_1"] = 100
payload["runs"][0]["N_2"] = 70
payload["runs"][0]["N_3"] = 30
payload["runs"][0]["P_1"] = 20
payload["runs"][0]["P_2"] = 20
payload["runs"][0]["P_3"] = 10
payload["runs"][0]["K_1"] = 20
payload["runs"][0]["K_2"] = 10
payload["runs"][0]["K_3"] = 10
payload["runs"][0]["NPK_T1"] = "06/04/2018"
payload["runs"][0]["NPK_T2"] = "15/05/2018"
payload["runs"][0]["NPK_T3"] = "15/06/2018"
print(payload)


Now the model can be run again with the custom fertilisation specified above:

In [ ]:
response = requests.post(URL, json=payload)
df = pd.DataFrame(response.json()["result"])
print(df.day)
print(df.TWSO)


With this level of fertilisation the harvest date moves forward by 2 days and yields increase from 317 to 1239 kg/ha
As mentioned above, it is possible to run multiple runs at once, as long as they're declared in the payload to be posted to the endpoint. The backend will deal with parallelisation.  
Let's say we want to look at the differences in yields just resulting by growing the same crop with the same management in different locations, say parcels 5646522, 3202428 and 2047584:

In [29]:
multiple_payloads = {
    "runs":
    [
        {
            "crop": "spring_barley",
            "year": 2018,
            "parcel_id": 5646522,
            "variety": "Spring_barley_301",
            "N_1": 100,
            "N_2": 70,
            "N_3": 30,
            "P_1": 20,
            "P_2": 20,
            "P_3": 10,
            "K_1": 20,
            "K_2": 10,
            "K_3": 10,
            "start_crop_calendar": "17/09/2017",
            "crop_start_date": "06/04/2018",
            "NPK_T1": "06/04/2018",
            "NPK_T2": "15/05/2018",
            "NPK_T3": "15/06/2018"
        },
        {
            "crop": "spring_barley",
            "year": 2018,
            "parcel_id": 3202428,
            "variety": "Spring_barley_301",
            "N_1": 100,
            "N_2": 70,
            "N_3": 30,
            "P_1": 20,
            "P_2": 20,
            "P_3": 10,
            "K_1": 20,
            "K_2": 10,
            "K_3": 10,
            "start_crop_calendar": "17/09/2017",
            "crop_start_date": "06/04/2018",
            "NPK_T1": "06/04/2018",
            "NPK_T2": "15/05/2018",
            "NPK_T3": "15/06/2018"
        },
        {
            "crop": "spring_barley",
            "year": 2018,
            "parcel_id": 2047584,
            "variety": "Spring_barley_301",
            "N_1": 100,
            "N_2": 70,
            "N_3": 30,
            "P_1": 20,
            "P_2": 20,
            "P_3": 10,
            "K_1": 20,
            "K_2": 10,
            "K_3": 10,
            "start_crop_calendar": "17/09/2017",
            "crop_start_date": "06/04/2018",
            "NPK_T1": "06/04/2018",
            "NPK_T2": "15/05/2018",
            "NPK_T3": "15/06/2018"
        }
    ]
}

response = requests.post(URL, json=multiple_payloads)
df = pd.DataFrame(response.json()["result"])
print(df.day)
print(df.TWSO)

0    2018-07-28
1    2018-07-28
2    2018-08-01
Name: day, dtype: object
0    1239.303542
1    1269.339514
2    1242.635874
Name: TWSO, dtype: float64


Here we find that yields vary between 1239 kg/ha and 1269 kg/ha depending on location and harvest dates range from July 28th, 2018 to August 1st, 2018 again depending on location.  
It is possible to pass as many runs at once as needed, and the backend will deal with parallelisation for speed. There can be many ways of generating runs; as an example, we are going to load a csv file where each row is an individual WOFOST run and each column represent a user-defined location, management parameter or initial soil/location state. 

In [35]:
input_runs = pd.read_csv("../resources/runs.csv")
print(input_runs.head())

         WAV     SMLIM    CRAIRC  WILTING_POTENTIAL  FIELD_CAPACITY      SOPE  \
0  28.993954  0.252104  0.059836           3.155055       -0.251808  5.136827   
1  32.571064  0.308811  0.071538           3.192978       -0.281784  6.929631   
2  16.051732  0.242196  0.077772           3.205634       -0.242963  4.527424   
3  42.242819  0.412980  0.065736           3.066218       -0.267952  3.717696   
4  40.764031  0.244113  0.054666           2.992970       -0.253597  3.225637   

       KSUB     RDMSOL    NAVAILI    PAVAILI  ...  year  parcel_id  iteration  \
0  6.907536  66.470943  17.205704  33.559347  ...  2021    2047584          1   
1  9.312907  77.221740  49.706963  33.062897  ...  2021    2047584          2   
2  6.149940  90.271393  35.798443  29.074013  ...  2021    2047584          3   
3  9.420882  83.082130  82.380311  21.512735  ...  2021    2047584          4   
4  3.027348  93.819636  29.605095  23.605997  ...  2021    2047584          5   

             variety  star

This can be then converted into a dictionary that follows the same format shown above:

In [36]:
csv_payload = {"runs": input_runs.to_dict(orient="records")}

This allows to now send a POST request to the `run_bulk` API endpoint as before. Note that the file contains 210 runs, so it will take some time to run, depending on the number of cores on your machine.

In [37]:
response = requests.post(URL, json=csv_payload)
csv_df = pd.DataFrame(response.json()["result"])

Once run, we can access the results as before using standart pandas syntax. The output dataframe will contain a new column called "run_id" which will allow to link the output with the input runs. For example, `run1` would be the first item in the JSON payload (i.e., the first row of the csv file).

In [38]:
print(csv_df.head())

          day       DVS       LAI         TAGP         TWSO         TWLV  \
0  2022-07-30  2.000000  0.397531  9101.203662  1694.221073  3017.094835   
1  2022-07-15  1.905064  0.082866  2770.116099   481.742067   791.506858   
2  2022-07-17  2.000000  0.267382  5059.280136  2029.741522  1119.759222   
3  2022-07-12  1.665726  0.118513  3705.013183    97.779615  1315.956217   
4  2022-07-25  2.000000  0.395663  8189.177052  2065.534068  2292.546426   

          TWST         TWRT       TRA         RD  ...  RKuptake  NamountSO  \
0  4389.887755  2477.147734  0.074775  77.221740  ...       0.0  27.060388   
1  1496.867174   943.711094  0.016554  66.470943  ...       0.0   7.839031   
2  1909.779392  1000.956354  0.046576  90.271393  ...       0.0  31.410046   
3  2291.277352  1542.778230  0.027190  93.819636  ...       0.0   1.629802   
4  3831.096557  1895.879278  0.066345  45.408654  ...       0.0  32.656798   

   PamountSO  KamountSO           crop  parcel_id  year            variety